# Tariikhna — Golden Example Cleanup

Takes the raw golden-story outputs and produces **defense-ready** versions by:
1. **Fixing character names** - normalizes mangled names against a canonical list
2. **Capping panels** - selects the best ~6-8 panels per story (drops near-duplicates)
3. **Expanding image prompts** - enriches each kept prompt to 250-300 words

No GPU needed - uses the base model via Groq for prompt expansion.

### Inputs needed:
- Your raw story JSONs (the_orphan_taken_in.json, abu_bakr_frees_the_slaves.json, etc.)
- A Groq API key

In [ ]:
!pip install groq -q
import json, re, time
from groq import Groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 2.5 MB/s eta 0:00:00


## 1. Canonical Name Map

The fine-tuned model sometimes mangles transliterated Arabic names. This maps the
wrong versions back to correct spellings. Add any others you spot.

In [ ]:
# Canonical spellings (correct) -> list of wrong/mangled variants seen in outputs
CANONICAL_NAMES = {
    'Abu Talib': [
        'Abū Ḥabīb', 'Abū Ḥakīm', 'Abū Ḥālib', 'Abū Ḥāṭib', 'Abū ḤāṭibҐ',
        'Abu al-Hasan', 'Abū Ṭālib', 'Abū ʿLālib', 'Ḥāṭib', 'Abu Hakim',
        'Abu Habib', 'Abu Hatib', 'Abu Halib'
    ],
    'Abd al-Muttalib': [
        'ʿ Abd al-Muḥd ibn al-Ḥaḍīd', 'ʿ Abd al-Muṭṭalib', 'Abd al-Muttalib',
        'ʿAbd al-Muṭṭalib', 'Abd al-Muhd ibn al-Hadid', 'al-Ḥaḍramah', 'Ḥaḍramah'
    ],
    'Muhammad': ['Muhammad', 'The Prophet Muhammad', 'Abd al-Muḥd'],
    'the Prophet': ['The Prophet'],
    'Abu Bakr': ['Abu Bakr', 'Abū Bakr'],
    'Bilal': ['Bilal', 'Bilāl'],
    'Umar ibn al-Khattab': ['Umar ibn al-Khatab', 'ʿ Umar', 'ʿUmar'],
    'Abu Jahl': ['Abu Jahl', 'Abū Jahl'],
    'Zunayrah': ['Zunayrah'],
    'al-Nahdiyah': ['Al-Nahdiyah', 'al-Nahdiyah'],
    'Umm Ubays': ['Umm Ubays', 'Umm ʿ Ubays'],
    'Amir ibn Fuhayrah': ["'Amir ibn Fuhayrah", 'ʿĀmir ibn Fuhayrah'],
    'al-Tufayl ibn Abdullah': ['al-Hafayl ibn Abdulla al-Asdi', 'al- Ṭufayl ibn ʿ Abdullāh al-Asdī'],
    'al-Aswad ibn Abd Yaghuth': ['al-Aswad ibn Abd Yaghuth', 'al-Aswad ibn ʿ Abd Yaghūth'],
    'Zayd': ['Zayd', 'Zayd ibn Harithah', 'Zayd ibn Ḥārithah'],
    'Khadijah': ['Khadijah', 'Khadījah'],
    'Safiyyah': ['Ṣafiyyah', 'Ṣafiyyah bint Ḥuyayy', 'Safiyyah'],
    'Bahira': ['Baḥīrā', 'Bahira'],
    'Halimah': ['Ḥalīmah', 'Halimah'],
}

# Build a reverse lookup: wrong -> correct
NAME_FIX = {}
for correct, variants in CANONICAL_NAMES.items():
    for v in variants:
        NAME_FIX[v.strip()] = correct
        # also normalized (collapse spaces) version
        NAME_FIX[re.sub(r'\s+', ' ', v).strip()] = correct

def fix_names_in_text(text):
    """Replace mangled names in a piece of text with canonical ones."""
    if not text:
        return text
    # Sort by length (longest first) so longer names replace before shorter substrings
    for wrong in sorted(NAME_FIX.keys(), key=len, reverse=True):
        if wrong and wrong in text:
            text = text.replace(wrong, NAME_FIX[wrong])
    return text

print(f'Name map ready: {len(NAME_FIX)} variant spellings -> canonical')

Name map ready: 56 variant spellings -> canonical


## 2. Groq Setup (for prompt expansion)

In [ ]:
GROQ_KEY = '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'
groq_client = Groq(api_key=GROQ_KEY)
GROQ_MODEL = 'llama-3.1-8b-instant'

def call_groq(system, user, max_tokens=1024, temp=0.5):
    for _ in range(5):
        try:
            r = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{'role':'system','content':system},{'role':'user','content':user}],
                max_tokens=max_tokens, temperature=temp)
            return r.choices[0].message.content
        except Exception as e:
            if '429' in str(e) or 'rate' in str(e).lower():
                print('  rate limited, waiting 30s'); time.sleep(30)
            else:
                print(f'  error: {str(e)[:80]}'); time.sleep(5)
    return None

print('Groq ready!')

Groq ready!


## 3. Panel Selection (cap to the best N)

Drops near-duplicate panels and keeps the key story beats. Uses the base model
to pick the most distinct, visually-compelling panels.

In [ ]:
# ============================================
# How many panels per story you want for the defense
# ============================================
PANELS_PER_STORY = 7

SELECT_SYSTEM = """You are an editor for a children's Islamic comic. Given a list of panels (with their narrative text), select the BEST ones that together tell the complete story clearly. Choose panels that are: distinct (not repetitive), visually compelling, and that cover the key story beats from beginning to end. Drop near-duplicate panels.

Output ONLY a JSON array of the panel_numbers to keep, in story order. Example: [1, 3, 5, 8, 11, 14, 17]"""

def select_best_panels(panels, n=PANELS_PER_STORY):
    """Pick the best n panels that tell the complete story."""
    if len(panels) <= n:
        return list(range(1, len(panels)+1))  # keep all
    # Build a compact summary for the selector
    summary = '\n'.join([f"Panel {p['panel_number']}: {p['narrative_text']}" for p in panels])
    user = f'Select the best {n} panels that tell this complete story:\n\n{summary}'
    raw = call_groq(SELECT_SYSTEM, user, max_tokens=256, temp=0.3)
    try:
        text = re.sub(r'^[^\[]*', '', raw)  # find the array
        text = re.sub(r'[^\]]*$', '', text)
        nums = json.loads(text)
        return nums[:n]
    except:
        # Fallback: evenly spaced selection
        step = len(panels) / n
        return [panels[int(i*step)]['panel_number'] for i in range(n)]

print(f'Panel selector ready (target: {PANELS_PER_STORY} per story)')

Panel selector ready (target: 7 per story)


## 4. Image Prompt Expansion (to 250-300 words)

In [ ]:
EXPAND_SYSTEM = """You are an expert at writing detailed image-generation prompts for a children's Islamic educational comic (watercolor storybook style, ages 6-12).

You will receive a short image prompt plus the character details. Expand it into a RICH 250-300 word prompt.

CRITICAL RULES:
- NEVER use the literal word 'prophet' - use 'the central figure seen from behind' or 'the man in the green cloak'
- Prophets shown ONLY from behind, never the face
- Women fully and modestly dressed (jilbab, khimar)
- No modern objects, historically accurate 7th century Arabia

Include ALL of: art style, specific 7th century setting with architecture/objects, each character's build/posture/clothing with SPECIFIC colors, the action/gesture/emotion, lighting (source/direction/quality), composition (shot type/angle), color palette. End with 'No modern objects, historically accurate 7th century scene.'

Keep character appearances CONSISTENT with the details given. Output ONLY the expanded prompt text (no JSON, no quotes)."""

def expand_prompt(short_prompt, characters):
    """Expand a short image prompt to 250-300 words using character details."""
    char_details = '\n'.join([
        f"- {c.get('name','?')}: {c.get('appearance','')} (depiction: {c.get('depiction_rule','FULL')})"
        for c in characters
    ])
    user = f"Short prompt:\n{short_prompt}\n\nCharacter details to keep consistent:\n{char_details}\n\nExpand to 250-300 words."
    expanded = call_groq(EXPAND_SYSTEM, user, max_tokens=600, temp=0.6)
    return expanded.strip() if expanded else short_prompt

print('Prompt expander ready!')

Prompt expander ready!


## 5. Full Cleanup Pipeline

In [ ]:
def cleanup_story(story, verbose=True):
    """Fix names, cap panels, expand prompts for one story."""
    panels = story.get('panels', [])
    if not panels:
        if verbose: print('  No panels - skipping (may need re-run)')
        return story

    # Step 1: Fix names in story context
    ctx = story.get('story_context') or {}
    for field in ['story_title', 'introduction', 'conclusion', 'moral_lesson']:
        if field in ctx:
            ctx[field] = fix_names_in_text(ctx[field])
    if 'key_figures' in ctx:
        ctx['key_figures'] = [fix_names_in_text(k) for k in ctx['key_figures']]

    # Step 2: Select best panels
    if verbose: print(f'  Selecting best {PANELS_PER_STORY} of {len(panels)} panels...')
    keep_numbers = select_best_panels(panels, PANELS_PER_STORY)
    if verbose: print(f'  Keeping panels: {keep_numbers}')
    kept = [p for p in panels if p['panel_number'] in keep_numbers]
    time.sleep(1)

    # Step 3: Fix names + expand prompts for kept panels
    cleaned_panels = []
    for i, panel in enumerate(kept):
        # Fix names everywhere
        panel['narrative_text'] = fix_names_in_text(panel.get('narrative_text', ''))
        for c in panel.get('characters', []):
            c['name'] = fix_names_in_text(c.get('name', ''))
            c['appearance'] = fix_names_in_text(c.get('appearance', ''))
        # Expand the image prompt
        if verbose: print(f'    Expanding prompt {i+1}/{len(kept)}...')
        old_wc = len(panel.get('image_prompt', '').split())
        expanded = expand_prompt(panel.get('image_prompt', ''), panel.get('characters', []))
        expanded = fix_names_in_text(expanded)
        panel['image_prompt'] = expanded
        panel['panel_number'] = i + 1  # renumber sequentially
        new_wc = len(expanded.split())
        if verbose: print(f'      {old_wc}w -> {new_wc}w')
        cleaned_panels.append(panel)
        time.sleep(1)

    story['panels'] = cleaned_panels
    story['story_context'] = ctx
    return story

print('Cleanup pipeline ready!')

Cleanup pipeline ready!


## 6. Run Cleanup on All Stories

In [ ]:
# Upload your raw story JSONs
from google.colab import files
print('Upload your story JSONs (the_orphan_taken_in.json, abu_bakr_frees_the_slaves.json, mercy_even_to_enemies.json, etc.):')
uploaded = files.upload()
story_files = list(uploaded.keys())
print(f'Got {len(story_files)} files')

Upload your story JSONs (the_orphan_taken_in.json, abu_bakr_frees_the_slaves.json, mercy_even_to_enemies.json, etc.):


Saving mercy_even_to_enemies.json to mercy_even_to_enemies.json
Saving abu_bakr_frees_the_slaves.json to abu_bakr_frees_the_slaves.json
Saving the_orphan_taken_in.json to the_orphan_taken_in.json
Got 3 files


In [ ]:
import os
os.makedirs('golden_clean', exist_ok=True)

cleaned_stories = []
for fname in story_files:
    if not fname.endswith('.json'):
        continue
    print(f'\n{"="*55}')
    print(f'CLEANING: {fname}')
    print(f'{"="*55}')
    with open(fname, encoding='utf-8') as f:
        story = json.load(f)
    cleaned = cleanup_story(story, verbose=True)
    cleaned_stories.append(cleaned)
    out_name = f'golden_clean/{story.get("passage_id", fname.replace(".json",""))}_clean.json'
    with open(out_name, 'w', encoding='utf-8') as f:
        json.dump(cleaned, f, ensure_ascii=False, indent=2)
    print(f'  Saved: {out_name} ({len(cleaned.get("panels", []))} panels)')

print(f'\n\nDONE! Cleaned {len(cleaned_stories)} stories.')


CLEANING: mercy_even_to_enemies.json
  Selecting best 7 of 19 panels...
  Keeping panels: [2, 3, 6, 7, 13, 17, 19]
    Expanding prompt 1/7...
      169w -> 373w
    Expanding prompt 2/7...
      135w -> 329w
    Expanding prompt 3/7...
      106w -> 317w
    Expanding prompt 4/7...
      78w -> 342w
    Expanding prompt 5/7...
      138w -> 323w
    Expanding prompt 6/7...
      92w -> 330w
    Expanding prompt 7/7...
      102w -> 319w
  Saved: golden_clean/mercy_even_to_enemies_clean.json (7 panels)

CLEANING: abu_bakr_frees_the_slaves.json
  Selecting best 7 of 46 panels...
  Keeping panels: [1, 3, 4, 5, 7, 10, 14]
    Expanding prompt 1/7...
      116w -> 323w
    Expanding prompt 2/7...
      165w -> 355w
    Expanding prompt 3/7...
      179w -> 335w
    Expanding prompt 4/7...
      144w -> 280w
    Expanding prompt 5/7...
      76w -> 322w
    Expanding prompt 6/7...
      141w -> 294w
    Expanding prompt 7/7...
      107w -> 351w
  Saved: golden_clean/abu_bakr_frees_the_sla

## 7. Review the Cleaned Result

In [ ]:
for story in cleaned_stories:
    ctx = story.get('story_context') or {}
    print(f'\n{"="*60}')
    print(f'{ctx.get("story_title", story.get("display_title", "?"))}')
    print(f'{"="*60}')
    print(f'Intro: {ctx.get("introduction", "")[:200]}')
    print(f'\n{len(story.get("panels", []))} panels:')
    for p in story.get('panels', []):
        wc = len(p['image_prompt'].split())
        print(f'\n  Panel {p["panel_number"]} ({wc}w prompt)')
        print(f'    Narration: {p["narrative_text"]}')
    print(f'\nLesson: {ctx.get("moral_lesson", "")}')


Bilal's Kindness Test
Intro: Imagine you're walking with your friends, and you come across a place where some people you don't know are being buried. How would you feel? Bilal, a close friend of the Prophet Muhammad, had to make 

7 panels:

  Panel 1 (373w prompt)
    Narration: the Prophet's cousin, Bilal, was joining the Muslim army to fight alongside the Prophet.

  Panel 2 (329w prompt)
    Narration: Bilal and the women set off on their journey, passing by the burial site of the Jews killed in the battle.

  Panel 3 (317w prompt)
    Narration: When Bilal saw the burial site, he began to cry. His cousin was surprised and asked why he was crying.

  Panel 4 (342w prompt)
    Narration: the Prophet approached Bilal, looking disapproving. He had heard that Bilal had been singing songs in the marketplace, which was not in line with the teachings of Islam.

  Panel 5 (323w prompt)
    Narration: Bilal felt sorry for his actions and wanted to make things right with the Prophet. He app

In [ ]:
import shutil
shutil.make_archive('golden_clean', 'zip', 'golden_clean')
from google.colab import files
files.download('golden_clean.zip')
print('Downloaded golden_clean.zip - defense-ready stories!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded golden_clean.zip - defense-ready stories!


## 8. NOTE: Re-running the Failed Zayd Story

The Zayd story came out with 0 panels (Layer 1a failed - likely a rate limit).
To fix it, go back to the **golden examples generation notebook** and re-run
just that one story:

```python
# In the generation notebook, after loading the model:
zayd = [p for p in GOLDEN_PASSAGES if p['passage_id'] == 'zayd_the_first_muslim_man'][0]
res = generate_golden_story(zayd['passage'], zayd['source'], verbose=True)
res['passage_id'] = zayd['passage_id']
res['display_title'] = zayd['display_title']
with open('zayd_the_first_muslim_man.json', 'w', encoding='utf-8') as f:
    json.dump(res, f, ensure_ascii=False, indent=2)
```

Then bring that JSON back here and run it through cleanup with the others.